# ResNet‑18 Feature Maps and Image Embedding

This notebook explains how a pretrained ResNet‑18 model transforms an image into a 512‑D embedding.
It includes feature‑map visualizations, an architecture diagram, and CSV export.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from matplotlib.patches import FancyArrowPatch

In [ ]:
device = torch.device('cpu')
print('Using device:', device)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Identity()
model.eval()

In [ ]:
feature_maps = {}

def hook_fn(name):
    def hook(module, inp, out):
        feature_maps[name] = out.detach().cpu()
    return hook

model.conv1.register_forward_hook(hook_fn('conv1'))
model.layer1.register_forward_hook(hook_fn('layer1'))
model.layer2.register_forward_hook(hook_fn('layer2'))
model.layer3.register_forward_hook(hook_fn('layer3'))
model.layer4.register_forward_hook(hook_fn('layer4'))

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
img_path = Path('data') / '25DIV3FAY_1002.jpg'
assert img_path.exists(), 'Image not found'

img = Image.open(img_path).convert('RGB')
x = transform(img).unsqueeze(0)

plt.imshow(img)
plt.axis('off')
plt.title('Input Image')
plt.show()

In [ ]:
with torch.no_grad():
    embedding = model(x).squeeze().numpy()

print('Embedding shape:', embedding.shape)

In [ ]:
def visualize_feature_maps(fmap, title, max_channels=16):
    fmap = fmap.squeeze(0)
    n = min(fmap.shape[0], max_channels)
    grid = math.ceil(math.sqrt(n))
    plt.figure(figsize=(8,8))
    for i in range(n):
        fm = fmap[i]
        fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-6)
        plt.subplot(grid, grid, i+1)
        plt.imshow(fm, cmap='viridis')
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_feature_maps(feature_maps['conv1'], 'conv1: edges & colors')
visualize_feature_maps(feature_maps['layer1'], 'layer1: textures')
visualize_feature_maps(feature_maps['layer2'], 'layer2: canopy patterns')
visualize_feature_maps(feature_maps['layer3'], 'layer3: structure')
visualize_feature_maps(feature_maps['layer4'], 'layer4: semantics')

## Full ResNet‑18 Architecture Diagram

In [ ]:
layer_order = [
    'conv1',
    'layer1.0.conv1','layer1.0.conv2','layer1.1.conv1','layer1.1.conv2',
    'layer2.0.conv1','layer2.0.conv2','layer2.1.conv1','layer2.1.conv2',
    'layer3.0.conv1','layer3.0.conv2','layer3.1.conv1','layer3.1.conv2',
    'layer4.0.conv1','layer4.0.conv2','layer4.1.conv1','layer4.1.conv2'
]

def draw_arrow(ax, x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle='->',linewidth=1))

def representative_maps(fmap, n=3):
    fmap = fmap.squeeze(0)
    idxs = np.linspace(0, fmap.shape[0]-1, n, dtype=int)
    return [(fmap[i]-fmap[i].min())/(fmap[i].max()-fmap[i].min()+1e-6) for i in idxs]

fig, ax = plt.subplots(figsize=(22,7))
ax.axis('off')
ax.set_xlim(0,30)
ax.set_ylim(0,8)

ax.imshow(img.resize((128,128)), extent=(0.5,2.5,4,6))
ax.text(1.5,3.5,'Input Image',ha='center')

px,py,x = 2.5,5.0,4.0
for lname in layer_order:
    fmap = feature_maps[lname.split('.')[0]]
    reps = representative_maps(fmap)
    for i,fm in enumerate(reps):
        ax.imshow(fm, extent=(x,x+1,1.5+i*1.1,2.5+i*1.1), cmap='viridis')
    ax.text(x+0.5,1.1,lname,ha='center',fontsize=7)
    draw_arrow(ax,px,py,x,3.5)
    px,py = x+1,3.5
    x+=1.4

plt.show()

In [ ]:
df_embedding = pd.DataFrame([embedding], columns=[f'emb_{i}' for i in range(512)])
df_embedding.insert(0,'image_id',img_path.name)
df_embedding

In [ ]:
df_embedding.to_csv('image_embedding.csv', index=False)
print('image_embedding.csv saved')